In [ ]:
from dotenv import find_dotenv, load_dotenv
from uuid import UUID
from db.repositories.videos import VideoRepository, AuthorVideoLoader
from db.conf import create_db_engine, get_async_session
from core.agents.series import VideoSeriesAgent
from core.agents.common import TemplateManager, TopicAgent, gemini_2_5_flash_lite, gemini_2_5_flash, TopicManager

load_dotenv(find_dotenv())

engine = create_db_engine()
db = get_async_session(engine)

topic_manager = TopicManager(db)
tpm_mgr = TemplateManager()
model_flash = gemini_2_5_flash()
model_lite = gemini_2_5_flash_lite()

topic_agent = TopicAgent(model_flash, tpm_mgr, topic_manager)
agent = VideoSeriesAgent(model_lite, tpm_mgr, topic_agent)


In [ ]:
author_id = UUID("8f37db5e-a7f6-11f0-8100-6fca046f2af4")

async with db() as session:
  video_repo = VideoRepository(session)
  video_loader = AuthorVideoLoader(author_id, max_videos=30)
  video_data = await video_loader.load(video_repo)

  result = await agent.run(video_data)

result